In [ ]:
### Learnt new

# .transform -> comme un OVER (PARTITION BY)
    df_reviews = pd.read_csv("../data/raw/reviews.csv", parse_dates=["date"])
    df["year_total"] = (
        df
        .groupby("year")["n_reviews"]
        .transform("sum")
    )

### The generic template
    import matplotlib.pyplot as plt # -> the engine: 
    import seaborn as sns           # -> You give it a long DataFrame and column names, and it handles the grouping
    wide_df.plot() # Si wide dataset ? 

    fig, ax = plt.subplots(figsize=(10, 5))                                 # always start with fig, ax = plt.subplots().
    sns.lineplot(data=<long_df>, x="<col>", y="<col>", hue="<col>", ax=ax)  # hue -> le grouping 
    ax.set_title("<the answer, as a sentence>")
    ax.set_xlabel("<label>")
    ax.set_ylabel("<label>")
    fig.text(0.01, -0.02, "Note: ...", fontsize=9)  # 0.01 et -0.02 : position dans le cadre du graphe
    fig.savefig("../docs/<name>.png", dpi=150, bbox_inches="tight")

# Which chart to use:
    # line for change over time;
    # bar to compare categories;
    # histogram or box plot for a distribution;
    # scatter for the relationship between two numeric variables.

# Others 
    # .all() 
    #   assert np.isclose(table_to_check["pct_of_year"],100,atol=0.15).all() -> The order is always: compare first, then reduce

In [ ]:
### Task 1: the seasonality chart

import pandas as pd 
df_reviews = pd.read_csv("../data/raw/reviews.csv", parse_dates=["date"])
df_reviews["year"]=df_reviews["date"].dt.year
df_reviews["month"]=df_reviews["date"].dt.month
df_reviews=df_reviews.loc[(df_reviews["year"].between(2022,2025)) | ((df_reviews["year"]==2026) & (df_reviews["month"]<9)) ]

df_long = (
    df_reviews
    .groupby(["year", "month"], as_index=False)
    .agg(n_reviews = ("listing_id","count"))
)

import matplotlib.pyplot as plt
import seaborn as sns   
fig, ax = plt.subplots(figsize=(10, 5))                                 
sns.lineplot(data=df_long, x="month", y="n_reviews", hue="year", ax=ax) 
ax.set_title("June and October peaks, Jan.2022 - Aug.2026")
ax.set_xlabel("Month")
ax.set_ylabel("Number of reviews")
fig.text(0.01, -0.02,"Note: (i) data for September 2026 is partially missing so it is excluded (ii) data for October-December 2026 is entirely missing.", fontsize=9)
fig.savefig("../docs/fig_reviews_seasonality.png", dpi=150, bbox_inches="tight")

In [ ]:
### Task 2: compare the shapes, not the levels

import pandas as pd 
df_reviews = pd.read_csv("../data/raw/reviews.csv", parse_dates=["date"])
df_reviews["year"]=df_reviews["date"].dt.year
df_reviews["month"]=df_reviews["date"].dt.month
df_reviews=df_reviews.loc[(df_reviews["year"].between(2022,2025)) | ((df_reviews["year"]==2026) & (df_reviews["month"]<9)) ]

# Long table for charts
df_long = (
    df_reviews
    .groupby(["year", "month"], as_index=False)
    .agg(n_reviews = ("listing_id","count"))
)

# Keep complete years only 
df_long = df_long.loc[df_long.groupby("year")["month"].transform("count")==12]

# Month's share of its year's review
df_long["pct_of_year"]=(
    df_long["n_reviews"]
    / (df_long.groupby("year")["n_reviews"].transform("sum"))
).mul(100)

# Checks 
table_to_check = df_long.groupby("year", as_index=False)["pct_of_year"].sum()
display(table_to_check)
import numpy as np 
assert np.isclose(table_to_check["pct_of_year"],100,atol=0.15).all() 

# Charts 
import matplotlib.pyplot as plt
import seaborn as sns   
fig, ax = plt.subplots(figsize=(10, 5))                                 
sns.lineplot(data=df_long, x="month", y="pct_of_year", hue="year", ax=ax) 
ax.set_title("Reviews peak in June and October every year (2022–2025)")
ax.set_xlabel("Month")
ax.set_ylabel("Share of reviews (%)")
fig.text(0.01, -0.02,"Note: 2026 data is excluded because it is incomplete.", fontsize=9)
fig.savefig("../docs/fig_reviews_seasonality_shares.png", dpi=150, bbox_inches="tight")

# 25min